# WPI Country Coverage Check\n",
    "Compares which countries appear in the World Port Index (WPI) dataset vs. the trade data.\n",
    "WPI's `Country Code` column contains full country names, compared directly against trade matrix columns."

In [8]:
import pandas as pd

WPI_PATH          = '../data/WPI.csv'
TRADE_MATRIX_PATH = '../data/all_trade_matrices/value_trade_matrix_all_transport_modes_TOTAL.csv'

In [9]:
# Load WPI — 'Country Code' column holds full country names
wpi = pd.read_csv(WPI_PATH, usecols=['Main Port Name', 'Country Code', 'Harbor Size', 'Latitude', 'Longitude'])
wpi.columns = ['portName', 'country_name', 'harborSize', 'lat', 'lon']
print(f'WPI: {len(wpi):,} ports, {wpi["country_name"].nunique()} unique countries')

# Load trade matrix country names (column headers)
trade_matrix = pd.read_csv(TRADE_MATRIX_PATH, nrows=0)
trade_countries = set(trade_matrix.columns.tolist())
trade_countries.discard('World')
print(f'Trade matrix: {len(trade_countries)} countries')

WPI: 3,805 ports, 195 unique countries
Trade matrix: 227 countries


In [10]:
# Maps WPI country names → trade data country names, to reconcile spelling/naming differences.
# Only entries where the two datasets use different names for the same country are needed here.
COUNTRY_NAME_CONVERTER = {
    # Name differences (abbreviations, punctuation)
    'Bosnia and Herzegovina':                              'Bosnia Herzegovina',
    'British Indian Ocean Territory':                      'Br. Indian Ocean Terr.',
    'British Virgin Islands':                              'Br. Virgin Isds',
    'Cayman Islands':                                      'Cayman Isds',
    'Christmas Island':                                    'Christmas Isds',
    'Cook Islands':                                        'Cook Isds',
    'Dominican Republic':                                  'Dominican Rep.',
    'Falkland Islands (Islas Malvinas)':                   'Falkland Isds (Malvinas)',
    'Federated States of Micronesia':                      'FS Micronesia',
    'Marshall Islands':                                    'Marshall Isds',
    'Norfolk Island':                                      'Norfolk Isds',
    'Northern Mariana Islands':                            'N. Mariana Isds',
    'Solomon Islands':                                     'Solomon Isds',
    'The Bahamas':                                         'Bahamas',
    'The Gambia':                                          'Gambia',
    'Turks and Caicos Islands':                            'Turks and Caicos Isds',
    'Wallis and Futuna':                                   'Wallis and Futuna Isds',

    # Official/UN name differences
    'Brunei':                                              'Brunei Darussalam',
    'Burma':                                               'Myanmar',
    'Caribbean Netherlands':                               'Bonaire',
    'Russia':                                              'Russian Federation',
    'South Korea':                                         'Rep. of Korea',
    'North Korea':                                         "Dem. People's Rep. of Korea",
    'Tanzania':                                            'United Rep. of Tanzania',
    'Turkey':                                              'TÃ¼rkiye',
    'Vietnam':                                             'Viet Nam',

    # Overseas territories / SARs grouped under parent country in trade data
    'Hong Kong':                                           'China, Hong Kong SAR',
    'Macau':                                               'China, Macao SAR',
    'Taiwan':                                              'Other Asia, nes',

    # Congos — WPI disambiguates with city name, trade data uses official names
    'Congo (Brazzaville)':                                 'Congo',
    'Congo (Kinshasa)':                                    'Dem. Rep. of the Congo',

    # Encoding differences (trade data has accented names)
    "Cote D'Ivoire":                                       "CÃ´te d'Ivoire",
    'Curacao':                                             'CuraÃ§ao',

    # Name variants with extra geographic qualifier
    'Saint Helena, Ascension, and Tristan da Cunha':       'Saint Helena',
    'Saint Vincent and the Grenadines (Windward Islands)': 'Saint Vincent and the Grenadines',
    'Sint Maarten':                                        'Saint Maarten',

    # United States
    'United States':                                       'USA',
}

# Apply converter: remap WPI country names to trade data equivalents
wpi['country_name'] = wpi['country_name'].replace(COUNTRY_NAME_CONVERTER)
print(f'Applied {len(COUNTRY_NAME_CONVERTER)} name mappings')

Applied 37 name mappings


In [11]:
wpi_countries = set(wpi['country_name'].dropna().unique())

in_both       = sorted(trade_countries & wpi_countries)
in_trade_only = sorted(trade_countries - wpi_countries)
in_wpi_only   = sorted(wpi_countries   - trade_countries)

print(f'Countries in BOTH trade data and WPI:           {len(in_both)}')
print(f'Countries in trade data ONLY (no WPI port):    {len(in_trade_only)}')
print(f'Countries in WPI ONLY (not in trade data):     {len(in_wpi_only)}')

Countries in BOTH trade data and WPI:           177
Countries in trade data ONLY (no WPI port):    50
Countries in WPI ONLY (not in trade data):     18


In [12]:
print('=== Trade data countries with NO port in WPI ===')
for c in in_trade_only:
    print(f'  - {c}')

=== Trade data countries with NO port in WPI ===
  - Afghanistan
  - Andorra
  - Anguilla
  - Armenia
  - Austria
  - Azerbaijan
  - Belarus
  - Bhutan
  - Bolivia (Plurinational State of)
  - Botswana
  - Burkina Faso
  - Burundi
  - Central African Rep.
  - Chad
  - Cocos Isds
  - Czechia
  - Eswatini
  - Ethiopia
  - Fr. South Antarctic Terr.
  - Hungary
  - Kazakhstan
  - Kyrgyzstan
  - Lao People's Dem. Rep.
  - Lesotho
  - Luxembourg
  - Malawi
  - Mali
  - Mongolia
  - Montserrat
  - Nepal
  - Niger
  - North Macedonia
  - Pitcairn
  - Rep. of Moldova
  - Rwanda
  - Saint BarthÃ©lemy
  - San Marino
  - Serbia
  - Slovakia
  - South Sudan
  - State of Palestine
  - Switzerland
  - Tajikistan
  - Tokelau
  - Turkmenistan
  - Uganda
  - Unnamed: 0
  - Uzbekistan
  - Zambia
  - Zimbabwe


In [15]:
print('=== WPI countries NOT in trade data ===')
for c in in_wpi_only:
    print(f'  - {c}')

=== WPI countries NOT in trade data ===
  - Antarctica
  - Faroe Islands
  - French Guiana
  - Guadeloupe
  - Guernsey
  - Isle of Man
  - Jersey
  - Johnson Atoll
  - Martinique
  - Midway Islands
  - Monaco
  - Puerto Rico
  - Reunion
  - South Georgia and South Sandwich Islands
  - Svalbard
  - U.S. Virgin Islands
  - Wake Island
  - Western Sahara


In [14]:
# For trade countries that DO have WPI ports, show the harbor sizes available
print('=== Harbor size breakdown for trade countries in WPI ===')
wpi_trade = wpi[wpi['country_name'].isin(trade_countries)].copy()
size_counts = (wpi_trade.groupby('country_name')['harborSize']
               .value_counts()
               .unstack(fill_value=0)
               .reindex(columns=['Large', 'Medium', 'Small', 'Very Small'], fill_value=0))
size_counts['Total'] = size_counts.sum(axis=1)
size_counts = size_counts.sort_values('Large', ascending=False)
print(size_counts.to_string())

=== Harbor size breakdown for trade countries in WPI ===
harborSize                        Large  Medium  Small  Very Small  Total
country_name                                                             
USA                                  21      38    132         475    666
Italy                                12      11     71          28    122
Japan                                11      26     54          71    162
United Kingdom                        7      24     67          86    184
Cuba                                  6       3     10           6     25
France                                6      12     22          26     66
China                                 5       9     25          27     66
Germany                               5       4     11          15     35
Australia                             5       8     24          29     66
Finland                               5       7     11          14     37
Egypt                                 5       2      8 